# 05 — Screening de transformaciones unitarias

Este notebook prueba **una transformación por vez**. Su métrica es AUC-ROC en validación cruzada sobre desarrollo. La validación final se mantiene intacta: solo la configuración que gane este screening debe pasar al notebook 03 para una evaluación formal.

No genera submission de Kaggle.

## Qué se prueba en esta segunda ronda

- `spline_mora`: spline cúbico de cinco nudos sobre mora.
- `spline_minutos`: spline cúbico de cinco nudos sobre minutos.
- `spline_mora_minutos`: ambos splines juntos, con cinco nudos.
- `spline_mora_3_nudos` y `spline_mora_7_nudos`: sensibilidad de la flexibilidad del spline de mora.

Los resultados anteriores mostraron una señal prometedora en spline de mora. Ahora se prueba su forma y su combinación con minutos. Se comparan ambos modelos porque el resultado observado en KNN también mejoró y debe verificarse, aunque los splines cambien la geometría de distancia.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

PROYECTO = Path.cwd().resolve()
if PROYECTO.name == 'notebooks':
    PROYECTO = PROYECTO.parent
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

from src import transformaciones as tr

importlib.reload(tr)
RANDOM_STATE = 2026
SCREENING_ID = datetime.now(ZoneInfo('America/Bogota')).strftime('%Y%m%d_%H%M%S')
PROCESSED = PROYECTO / 'data' / 'processed'
OUTPUTS = PROYECTO / 'outputs' / 'experimentos'
OUTPUTS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

## 1. Fijar exactamente los mismos clientes de desarrollo

La partición se genera una vez con la misma semilla. Así, toda diferencia observada entre configuraciones procede de la transformación, no de clientes diferentes.

In [ ]:
train_original = pd.read_csv(PROCESSED / 'train_preparado.csv')
indices = np.arange(len(train_original))
idx_dev, idx_val = train_test_split(
    indices, test_size=0.20, stratify=train_original['resultado'], random_state=RANDOM_STATE
)

# idx_val queda reservado. En este screening nunca se usa para elegir una transformación.
display(pd.DataFrame({
    'grupo': ['desarrollo para CV', 'validación final reservada'],
    'clientes': [len(idx_dev), len(idx_val)]
}))

## 2. Definir los experimentos

Esta ronda conserva baseline como control y modifica únicamente la representación spline. El número de nudos controla flexibilidad: menos nudos equivale a una curva más suave; más nudos permite más curvatura, pero puede sobreajustar.


In [ ]:
EXPERIMENTOS = [
    ('baseline', 'Regresión logística'),
    ('baseline', 'KNN'),
    ('spline_mora', 'Regresión logística'),
    ('spline_mora', 'KNN'),
    ('spline_minutos', 'Regresión logística'),
    ('spline_minutos', 'KNN'),
    ('spline_mora_minutos', 'Regresión logística'),
    ('spline_mora_minutos', 'KNN'),
    ('spline_mora_3_nudos', 'Regresión logística'),
    ('spline_mora_3_nudos', 'KNN'),
    ('spline_mora_7_nudos', 'Regresión logística'),
    ('spline_mora_7_nudos', 'KNN'),
]

CV_SCREENING = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
PARAMETROS_LOGISTICA = {'modelo__C': [0.03, 0.1, 0.3, 1, 3, 10]}
PARAMETROS_KNN = {'modelo__n_neighbors': list(range(15, 82, 2)), 'modelo__weights': ['distance']}


## 3. Ejecutar la comparación controlada

Cada pipeline aprende imputación, escalamiento, codificación y, si aplica, splines, únicamente dentro de sus folds de entrenamiento. Los resultados no son todavía el resultado final del proyecto: son un filtro para decidir qué configuración merece pasar al notebook 03.

In [ ]:
def evaluar(configuracion_nombre, modelo_nombre):
    configuracion = tr.obtener_configuracion(configuracion_nombre)
    datos = tr.aplicar_transformaciones(train_original, configuracion)
    numericas, splines, categoricas = tr.obtener_grupos(configuracion)
    predictores = numericas + splines + categoricas

    X_dev = datos.iloc[idx_dev][predictores]
    y_dev = datos.iloc[idx_dev]['resultado']
    preprocesador = tr.crear_preprocesador(configuracion)

    if modelo_nombre == 'Regresión logística':
        pipeline = Pipeline([
            ('preprocesar', preprocesador),
            ('modelo', LogisticRegression(max_iter=3000, solver='liblinear', random_state=RANDOM_STATE))
        ])
        parametros = PARAMETROS_LOGISTICA
    else:
        pipeline = Pipeline([
            ('preprocesar', preprocesador),
            ('modelo', KNeighborsClassifier())
        ])
        parametros = PARAMETROS_KNN

    busqueda = GridSearchCV(
        pipeline, parametros, scoring='roc_auc', cv=CV_SCREENING, n_jobs=-1, refit=False, return_train_score=False
)
    busqueda.fit(X_dev, y_dev)
    return {
        'screening_id': SCREENING_ID,
        'configuracion': configuracion_nombre,
        'modelo': modelo_nombre,
        'transformaciones': str(configuracion),
        'mejores_parametros': str(busqueda.best_params_),
        'auc_cv_media': busqueda.best_score_,
        'desv_auc_cv': busqueda.cv_results_['std_test_score'][busqueda.best_index_],
        'n_predictores_antes_codificacion': len(predictores),
        'nudos_spline': configuracion.get('n_knots', pd.NA),
    }

resultados = pd.DataFrame([evaluar(configuracion, modelo) for configuracion, modelo in EXPERIMENTOS])
resultados = resultados.sort_values(['modelo', 'auc_cv_media'], ascending=[True, False]).reset_index(drop=True)
display(resultados.style.format({'auc_cv_media': '{:.4f}', 'desv_auc_cv': '{:.4f}'}))

## 4. Interpretar y registrar

Una mejora pequeña debe leerse junto con la desviación entre folds. Elegiremos como candidato únicamente una configuración que supere el baseline de su mismo modelo de forma clara y que conserve sentido de negocio. Después se ejecuta el notebook 03 solo para ese candidato, con CV repetida y validación final.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=resultados, x='auc_cv_media', y='configuracion', hue='modelo', ax=ax)
ax.set(xlabel='AUC-ROC medio en CV de desarrollo', ylabel='Configuración', title='Screening: una transformación por vez')
ax.legend(title='Modelo')
plt.tight_layout(); plt.show()

ARCHIVO_SCREENING = OUTPUTS / 'registro_screening_transformaciones.csv'
if ARCHIVO_SCREENING.exists():
    historico = pd.read_csv(ARCHIVO_SCREENING)
    historico = pd.concat([historico, resultados], ignore_index=True)
    historico = historico.drop_duplicates(subset=['screening_id', 'configuracion', 'modelo'], keep='last')
else:
    historico = resultados
historico.to_csv(ARCHIVO_SCREENING, index=False)

ganadores = resultados.groupby('modelo', as_index=False).first()[['modelo', 'configuracion', 'auc_cv_media', 'mejores_parametros']]
display(ganadores.style.format({'auc_cv_media': '{:.4f}'}))

## Qué hacer después

1. Comparar cada resultado contra el baseline del mismo modelo.
2. Si una configuración es estable y mejora el baseline, poner su nombre en `CONFIGURACION_ACTIVA` dentro de `src/transformaciones.py`.
3. Ejecutarla en el notebook 03 para validación repetida, calibración, odds ratios y registro formal.
4. Generar submission solo después de confirmar el resultado del notebook 03.
